# Batch Model Monitoring Demo - Batch Inference

Demonstrates a batch scoring pipeline:
1. Load the model from the registry
2. Run warehouse-based batch inference on all customers
3. Write results to SCORING_DATA (the monitor's source table)
4. Copy the first batch into BASELINE_DATA (used for drift comparison)

In [ ]:
!pip install --upgrade "snowflake-ml-python>=1.7.1" snowflake-connector-python

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()
session.use_schema("ML_DEMOS.BATCH_MONITORING")
session.use_warehouse("ML_DEMO_WH")

registry = Registry(session=session, database_name="ML_DEMOS", schema_name="BATCH_MONITORING")
mv = registry.get_model("CHURN_PREDICTOR").version("V1")
print(f"Model loaded: {mv.model_name} / {mv.version_name}")
print(f"Functions: {mv.show_functions()}")

In [ ]:
# Build scoring input from CUSTOMERS table
import pandas as pd

FEATURE_COLUMNS = [
    "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
    "NUM_SUPPORT_TICKETS", "DAYS_SINCE_LAST_LOGIN", "AVG_MONTHLY_USAGE_HOURS",
    "AGE", "CONTRACT_TYPE", "PLAN_TYPE",
]

customers_df = session.table("CUSTOMERS").to_pandas()
scoring_input = customers_df[["CUSTOMER_ID"] + FEATURE_COLUMNS].copy()
print(f"Scoring cohort: {len(scoring_input)} customers")
scoring_input.head()

In [ ]:
# Run batch inference on warehouse
features_only = scoring_input[FEATURE_COLUMNS]
sp_features = session.create_dataframe(features_only)

predictions = mv.run(sp_features, function_name="predict_proba")
pred_pd = predictions.to_pandas()

print(f"Predictions shape: {pred_pd.shape}")
print(f"Columns: {pred_pd.columns.tolist()}")
pred_pd.head()

In [ ]:
# Assemble SCORING_DATA records
import uuid
from datetime import datetime

# The predict_proba output has columns for each class - take the churn probability (class 1)
# Column name may vary - inspect and select the right one
proba_cols = [c for c in pred_pd.columns if "1" in c or "proba" in c.lower()]
if proba_cols:
    score_col = proba_cols[0]
else:
    score_col = pred_pd.columns[-1]

print(f"Using score column: {score_col}")

scoring_results = scoring_input.copy()
scoring_results["PREDICTION_SCORE"] = pred_pd[score_col].values
scoring_results["ID"] = [str(uuid.uuid4()) for _ in range(len(scoring_results))]
scoring_results["PREDICTION_TS"] = datetime(2025, 1, 15, 8, 0, 0).strftime("%Y-%m-%d %H:%M:%S")

# Reorder columns to match table schema
output_cols = [
    "ID", "CUSTOMER_ID", "PREDICTION_TS",
    "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
    "NUM_SUPPORT_TICKETS", "DAYS_SINCE_LAST_LOGIN", "AVG_MONTHLY_USAGE_HOURS",
    "AGE", "CONTRACT_TYPE", "PLAN_TYPE", "PREDICTION_SCORE",
]
scoring_results = scoring_results[output_cols]

print(f"\nScoring results shape: {scoring_results.shape}")
print(f"Score distribution:")
print(scoring_results["PREDICTION_SCORE"].describe())

In [ ]:
# Write to SCORING_DATA and BASELINE_DATA
sp_results = session.create_dataframe(scoring_results)
sp_results.write.mode("overwrite").save_as_table("SCORING_DATA")
sp_results.write.mode("overwrite").save_as_table("BASELINE_DATA")

print("Written to SCORING_DATA and BASELINE_DATA")
print(f"SCORING_DATA rows: {session.table('SCORING_DATA').count()}")
print(f"BASELINE_DATA rows: {session.table('BASELINE_DATA').count()}")

In [ ]:
%%sql -r df_check
SELECT
    MIN(prediction_score) AS min_score,
    AVG(prediction_score) AS avg_score,
    MAX(prediction_score) AS max_score,
    COUNT(*) AS total_rows
FROM SCORING_DATA;

### Alternative: Pure SQL batch inference

If running outside of a Python notebook, you can use SQL directly:
```sql
INSERT INTO SCORING_DATA
SELECT
    UUID_STRING() AS id, customer_id,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS prediction_ts,
    tenure_months, monthly_charges, total_charges,
    num_support_tickets, days_since_last_login, avg_monthly_usage_hours,
    age, contract_type, plan_type,
    MODEL(CHURN_PREDICTOR, V1)!PREDICT_PROBA(
        tenure_months, monthly_charges, total_charges,
        num_support_tickets, days_since_last_login, avg_monthly_usage_hours,
        age, contract_type, plan_type
    ):output_feature_1::FLOAT AS prediction_score
FROM CUSTOMERS;
```